In [2]:
# !pip install --q --upgrade trl
!pip install --upgrade accelerate
!pip install autoawq
!pip install bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for autoawq: filename=autoawq-0.2.9-py3-none-any.whl size=115106 sha256=b7ae80ff417e2c482d6feaf779ed23525a1f763505d39fbc05a62e5a1a0922af
  Stored in directory: /root/.cache/pip/wheels/45/1a/7b/7314b3a958454e8ce349f600829a3f0a6a05aeebf987be1e16
Successfully built autoawq
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.8 MB/s eta 0:00:00


In [32]:
from datasets import load_dataset
from datasets import concatenate_datasets
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments

In [4]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

In [5]:
import bitsandbytes as bnb

print(bnb.__version__)

0.48.2


In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

In [8]:
import torch
print("Allocated:", torch.cuda.memory_allocated()/1024**3, "GB")
print("Reserved:", torch.cuda.memory_reserved()/1024**3, "GB")

Allocated: 5.176027774810791 GB
Reserved: 6.650390625 GB


In [9]:
print(next(model.model.layers[0].parameters()).dtype)

torch.uint8


In [10]:
ds_1 = load_dataset("chillies/IELTS-writing-task-2-evaluation")
ds_2 = load_dataset("chillies/IELTS_evaluations")


train.csv:   0%|          | 0.00/44.5M [00:00<?, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9833 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/491 [00:00<?, ? examples/s]

train.csv:   0%|          | 0.00/48.2M [00:00<?, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9912 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/495 [00:00<?, ? examples/s]

In [11]:
ds_train = concatenate_datasets({ds_1["train"], ds_2["train"]})
ds_test = concatenate_datasets({ds_1["test"], ds_2["test"]})
print(ds_train)
print(ds_test)

Dataset({
    features: ['prompt', 'essay', 'evaluation', 'band'],
    num_rows: 19745
})
Dataset({
    features: ['prompt', 'essay', 'evaluation', 'band'],
    num_rows: 986
})


In [41]:
prompt_template_with_input = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}
"""

instruction_prompt ="""You are an IELTS Writing Task 2 examiner.
 Evaluate the essay below in detail according to the four official IELTS Writing criteria: Task Achievement, Coherence and Cohesion, Lexical Resource, and Grammatical Range and Accuracy.
 For each criterion, provide specific comments, examples, and a suggested band score.
 Then, provide an Overall Band Score and general feedback including Strengths, Areas for Improvement, and Suggestions for Enhancement.
 """

print(prompt_template_with_input.format(instruction = instruction_prompt,
                                        input = f"Essay Topic:\n {ds_train[0]["prompt"]}\n\nEssay:\n{ds_train[0]["essay"]} ",
                                        output =  f"Evaluation:\n{data["evaluation"]}\n\nOverall Band Score :\n{data["band"]}"
                                        ))

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an IELTS Writing Task 2 examiner.
 Evaluate the essay below in detail according to the four official IELTS Writing criteria: Task Achievement, Coherence and Cohesion, Lexical Resource, and Grammatical Range and Accuracy.
 For each criterion, provide specific comments, examples, and a suggested band score.
 Then, provide an Overall Band Score and general feedback including Strengths, Areas for Improvement, and Suggestions for Enhancement.
 

### Input:
Essay Topic:
 Young people who commit crimes should be treated in the same way as adults who commit crimes.

To what extent do you agree or disagree?

Essay:
Deciding to choose among the potential ways of punishing young people who commit crimes continues to be a controversial issue for the societies and the governments. It is argued by some that these peopl

In [43]:
process_data = []
for data in ds_train :
  formatted_input = prompt_template_with_input.format(instruction = instruction_prompt,
                                                      input = f"Essay Topic:\n {data["prompt"]}\n\nEssay:\n{data["essay"]} ",
                                                      output =  f"Evaluation:\n{data["evaluation"]}\n\nOverall Band Score :\n{data["band"]}"
                                                      )
  process_data.append({"text": formatted_input, "output" : f"Evaluation:\n{data["evaluation"]}\n\nOverall Band Score :\n{data["band"]}"})


In [46]:
process_data[1]

{'text': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nYou are an IELTS Writing Task 2 examiner.\n Evaluate the essay below in detail according to the four official IELTS Writing criteria: Task Achievement, Coherence and Cohesion, Lexical Resource, and Grammatical Range and Accuracy.\n For each criterion, provide specific comments, examples, and a suggested band score.\n Then, provide an Overall Band Score and general feedback including Strengths, Areas for Improvement, and Suggestions for Enhancement.\n \n\n### Input:\nEssay Topic:\n Young people who commit crimes should be treated in the same way as adults who commit crimes. To what extend do you agree or disagree?\n\nEssay:\nIn this modern era, youngsters who commit offences should be punished in  similar ways as mature people. I do think that younger criminals should have behaved less strictly than o

In [17]:
!pip install peft
!pip install trl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 9.4 MB/s eta 0:00:00


In [23]:
from peft import LoraConfig , get_peft_model , prepare_model_for_kbit_training
from trl import SFTTrainer
from sklearn.model_selection import train_test_split


In [47]:
from datasets import Dataset
train_indices, val_indices = train_test_split(range(len(process_data)), test_size=0.1, random_state=42)
process_dataset = Dataset.from_list(process_data)
train_dataset = process_dataset.select(train_indices)
val_dataset = process_dataset.select(val_indices)

In [49]:
lora_config = LoraConfig (
  r =8 , # Rank
  lora_alpha =32 , # Alpha parameter
  target_modules =[ # Target attention matrices
  "q_proj",
  "k_proj",
  "v_proj",
  "o_proj"
  ],
  lora_dropout =  0.05,
  bias = "none",
  task_type = "CAUSAL_LM"
)


In [50]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 5,046,272 || all params: 7,620,662,784 || trainable%: 0.0662


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [34]:
training_args = TrainingArguments(
  output_dir = "./output",
  num_train_epochs = 2 ,
  per_device_train_batch_size = 4,
  gradient_accumulation_steps= 4,
  learning_rate = 1e-4,
  fp16=True,
  save_total_limit = 3,
  logging_steps = 1,
  optim="paged_adamw_8bit"
)

In [51]:
trainer = SFTTrainer(
  model=model,
  train_dataset=train_dataset,
  # tokenizer=tokenizer, # Removed the tokenizer argument
  args=training_args,
  # max_seq_length=4096,
  eval_dataset=val_dataset,
)
trainer.train()

Adding EOS to train dataset:   0%|          | 0/17770 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/17770 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/17770 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1975 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1975 [00:00<?, ? examples/s]

...

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: trinhphuthai-2k5 (trinhphuthai-2k5-hanoi-university-of-science-and-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.32 GiB. GPU 0 has a total capacity of 14.74 GiB of which 400.12 MiB is free. Process 11661 has 14.35 GiB memory in use. Of the allocated memory 13.39 GiB is allocated by PyTorch, and 855.86 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)